## 0. Setup shared by the three validation notebooks
Paths are relative to the repository root. The statistics come from `tools/validation_metrics.py`
(Ministry of Transport guideline, draft 6, September 2024); results go to `Output/validation_mot/`.

In [1]:
import os, sys, warnings
while not os.path.exists('METHODOLOGY.md') and os.getcwd() != '/': os.chdir('..')
assert os.path.exists('METHODOLOGY.md'), 'run from inside the Nofit_LRT_Extension repository'
sys.path.insert(0, 'tools')
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import validation_metrics as vm
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)

BLUE, ORANGE, AQUA, PURPLE = '#2a78d6', '#eb6834', '#1baf7a', '#7b5bd6'
INK, INK2, MUTED, GRID, AXIS = '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7'
OUT = 'Output/validation_mot'; FIG = 'Output/figures'
# Time window (wave 1b): AM 06:00-09:00 is the base and the default; PM 16:00-19:00 and midday 10:00-14:00 with the environment variable NOFIT_PERIOD
PERIOD = os.environ.get('NOFIT_PERIOD', 'AM')
HOURS, SUF = {'AM': ([6, 7, 8], ''), 'PM': ([16, 17, 18], '_pm'), 'MD': ([10, 11, 12, 13], '_md')}[PERIOD]
PLABEL = {'AM': 'AM 06:00-09:00', 'PM': 'PM 16:00-19:00', 'MD': 'midday 10:00-14:00'}[PERIOD]
W = {'AM': '06-09', 'PM': '16-19', 'MD': '10-14'}[PERIOD]
PSUF = '' if PERIOD == 'AM' else '_' + PERIOD            # suffix of the validation output files of the other periods
def vfile(name): return f'{OUT}/{name}{PSUF}.csv'
os.makedirs(OUT, exist_ok=True); os.makedirs(FIG, exist_ok=True)

def style_ax(ax, title=None, xlabel=None, ylabel=None):
    ax.grid(True, color=GRID, linewidth=0.6); ax.set_axisbelow(True)
    for s in ax.spines.values(): s.set_color(AXIS)
    ax.tick_params(colors=INK2, labelsize=9)
    if title: ax.set_title(title, color=INK, fontsize=11)
    if xlabel: ax.set_xlabel(xlabel, color=INK2, fontsize=10)
    if ylabel: ax.set_ylabel(ylabel, color=INK2, fontsize=10)

def is_pointer(p): return open(p, 'rb').read(40).startswith(b'version https://git-lfs')
def need(*paths):
    for p in paths: assert not is_pointer(p), f'{p} is a Git LFS pointer: python3 tools/lfs_pull.py {p}'

SUMMARY = []      # one row per guideline check; written to Output/validation_mot/summary_<stage>.csv
def add_summary(test, ref, item, level, criterion, result, status, explanation, data):
    SUMMARY.append({'period': PERIOD, 'test': test, 'guideline': ref, 'item': item, 'level': level, 'criterion': criterion,
                    'result': result, 'status': status, 'explanation': explanation, 'data (vintage)': data})

def scatter_origin(ax, obs, mod, title, xlabel, ylabel, logscale=False, color=BLUE):
    """Guideline format: observed on x, model on y, regression through the origin with equation and R2."""
    o = np.asarray(obs, float); m = np.asarray(mod, float); ok = np.isfinite(o) & np.isfinite(m)
    o, m = o[ok], m[ok]
    ax.scatter(o, m, s=10, color=color, alpha=0.55, linewidths=0)
    hi = max(o.max(), m.max()) * 1.05 if len(o) else 1
    ax.plot([0, hi], [0, hi], color=MUTED, lw=0.8, ls=':')
    b = vm.slope_origin(o, m); r2 = vm.r2_correl(o, m)
    ax.plot([0, hi], [0, b * hi], color=ORANGE, lw=1.5)
    ax.text(0.04, 0.95, f'y = {b:.2f}x   R² = {r2:.2f}   n = {len(o)}', transform=ax.transAxes, va='top', fontsize=9, color=INK)
    if logscale: ax.set_xscale('symlog', linthresh=10); ax.set_yscale('symlog', linthresh=10)
    style_ax(ax, title, xlabel, ylabel)

def save_fig(fig, name):
    fig.savefig(f'{FIG}/{name.replace(".png", PSUF + ".png")}', dpi=150, bbox_inches='tight', facecolor='white'); plt.close(fig)

# Ministry of Transport validation, stage 2: timing (T8)

**Question.** Does the survey's time-of-day profile agree with profiles measured independently of it?

**Test (guideline 4.1).** Coincidence ratio (CR) between the survey's departure-time distribution and (a) the hourly
traffic-count profile on the links crossing each of the six closed cordons, 06:00-19:00, for car trips with exactly one end
inside the cordon, by direction; (b) the RavKav 2025 boarding profile by quarter-hour, 06:00-09:00, for bus and Metronit.
**Criterion:** CR above 0.6. The guideline compares the survey profile with the survey itself; here the comparison data are
independent. Survey trips: both ends in the study area, departure time = the trip's start time (`STStr`, minute resolution),
expanded by `new_wf`. Time gaps: survey 2017/18, counts dated 2017-2023, RavKav 2025; profiles are shapes (shares), so no growth is applied.

In [2]:
TRIPS = 'Input/THS_2017-2018/trips_ths_2017.xlsx'
KEYS = 'Input/Matrices/1270_02_09_2021_TAZ_North_keys.csv'
need(TRIPS, KEYS, 'Input/TAZ_2636_Keys.xlsx')
zon = pd.read_csv('Input/Zonal_2020.csv', encoding='cp1255').set_index('TAZ_ID')
TAZ = [int(c) for c in pd.read_csv('Output/ths2017/two_mode/car_taz.csv', index_col=0).columns]
keys_raw = pd.read_csv(KEYS, encoding='windows-1255')
_k = keys_raw[['TAZ_NUMBER', 'SZ_NEW']].dropna().astype(int).drop_duplicates('TAZ_NUMBER').set_index('TAZ_NUMBER')['SZ_NEW']
taz_sz = _k.reindex(TAZ).astype(int)
kd = keys_raw[['TAZ_1270', 'TAZ_NUMBER']].dropna(subset=['TAZ_NUMBER']).astype({'TAZ_NUMBER': int}).drop_duplicates('TAZ_NUMBER')
taz_1250 = kd.set_index('TAZ_NUMBER')['TAZ_1270'].reindex(TAZ); Z1250 = sorted(taz_1250.unique())
d_ = pd.DataFrame({'z': taz_1250.values, 'g': taz_sz.values, 'pop': zon.loc[TAZ, 'POPULATION'].values}, index=TAZ)
sz_of_1250 = pd.Series({z: int(g.groupby('g')['pop'].sum().idxmax()) for z, g in d_.groupby('z')})

df = pd.read_excel(TRIPS)
d = df.sort_values(['PerID3', 'SurveyDay', 'placeno']).copy()
g = d.groupby(['PerID3', 'SurveyDay'])
d['origin'] = g['actTaz'].shift(1)
trips = d.dropna(subset=['origin', 'actTaz', 'STStr']).copy()
k26 = pd.read_excel('Input/TAZ_2636_Keys.xlsx').drop_duplicates('TAZ_2636').set_index('TAZ_2636')['TAZ_1250']
trips['o1250'] = trips['origin'].map(k26); trips['d1250'] = trips['actTaz'].map(k26)
S = trips[trips['o1250'].isin(Z1250) & trips['d1250'].isin(Z1250)].copy()
S['sz_o'] = S['o1250'].map(sz_of_1250); S['sz_d'] = S['d1250'].map(sz_of_1250)
S['t'] = S['STStr'] % 24; S['w'] = S['new_wf'] / 2
S['grp'] = S['mainmode'].map(lambda m: 'car' if m in (10, 11) else 'bus' if m in (3, 5) else 'metronit' if m == 5 else 'taxi' if m in (4, 8) else 'rail' if m == 7 else 'other')
print(f'survey trips, both ends in the study area, with a start time: {len(S):,} sampled; share of the day departing 06:00-09:00: {S.loc[(S.t >= 6) & (S.t < 9), "w"].sum() / S.w.sum():.3f}')

survey trips, both ends in the study area, with a start time: 106,770 sampled; share of the day departing 06:00-09:00: 0.243


## 1. Car: survey departures against the cordon counts

In [3]:
CORDONS = {'Haifa city': [8, 12, 13, 14, 15, 16, 17], 'Krayot': [5, 6, 7], 'Kiryat Ata + Zevulun': [9], 'Nazareth + Nof HaGalil': [19, 20],
           'Tirat Carmel': [24], 'Metropolitan core': [5, 6, 7, 8, 9, 12, 13, 14, 15, 16, 17, 18]}
cnt = pd.read_csv('Output/validation/car_cordon_count_hourly_profile.csv')
cnt = cnt.replace({'Metropolitan core (Haifa, Nesher, Krayot, Kiryat Ata)': 'Metropolitan core'})
HOURS = list(range(6, 20)); hcols = [f'YARAM{h}' for h in HOURS]
car = S[S.grp == 'car']
rows = []; PROF = {}
for name, szs in CORDONS.items():
    o_in = car.sz_o.isin(szs); d_in = car.sz_d.isin(szs)
    for direction, sel in [('inbound', ~o_in & d_in), ('outbound', o_in & ~d_in)]:
        x = car[sel]
        sv = np.array([x.loc[(x.t >= h) & (x.t < h + 1), 'w'].sum() for h in HOURS]); sv_share = sv / sv.sum()
        cr = cnt[(cnt.cordon == name) & (cnt.direction == direction)]
        if cr.empty: continue
        c_share = cr[hcols].values.ravel().astype(float); c_share = c_share / c_share.sum()
        am_s, am_c = sv_share[:3], c_share[:3]
        rows.append({'cordon': name, 'direction': direction, 'survey trips (sampled)': len(x), 'CR 06:00-19:00': vm.coincidence_ratio(sv_share, c_share),
                     'CR 06:00-09:00 (3 hours)': vm.coincidence_ratio(am_s, am_c), 'survey AM share of 06-19': float(sv_share[:3].sum()), 'count AM share of 06-19': float(c_share[:3].sum()),
                     'survey busiest AM clock hour / 3 h': float(sv_share[:3].max() / sv_share[:3].sum()), 'count busiest AM clock hour / 3 h': float(c_share[:3].max() / c_share[:3].sum())})
        PROF[(name, direction)] = (sv_share, c_share)
# pooled over all cordon-directions: sum of the survey crossings, mean of the count shares weighted by the AM counts is not available, so use the unweighted mean of the shares
car_prof = pd.DataFrame(rows)
car_prof.to_csv(f'{OUT}/T8_car_profile_vs_counts.csv', index=False, float_format='%.4f'); car_prof.round(3)

,cordon,direction,survey trips (sampled),CR 06:00-19:00,CR 06:00-09:00 (3 hours),survey AM share of 06-19,count AM share of 06-19,survey busiest AM clock hour / 3 h,count busiest AM clock hour / 3 h
0,Haifa city,inbound,3311,0.776,0.723,0.345,0.223,0.449,0.419
1,Haifa city,outbound,3326,0.772,0.730,0.155,0.194,0.522,0.426
2,Krayot,inbound,1558,0.754,0.712,0.208,0.220,0.517,0.420
3,Krayot,outbound,1560,0.771,0.733,0.273,0.165,0.522,0.442
4,Kiryat Ata + Zevulun,inbound,757,0.786,0.821,0.182,0.203,0.378,0.368
5,Kiryat Ata + Zevulun,outbound,764,0.707,0.707,0.326,0.198,0.540,0.393
6,Nazareth + Nof HaGalil,inbound,662,0.664,0.781,0.179,0.192,0.435,0.407
7,Nazareth + Nof HaGalil,outbound,663,0.633,0.639,0.398,0.279,0.476,0.358
8,Tirat Carmel,inbound,468,0.733,0.614,0.215,0.230,0.504,0.461
9,Tirat Carmel,outbound,478,0.771,0.770,0.304,0.231,0.483,0.405


In [4]:
for _, r in car_prof.iterrows():
    ok = r['CR 06:00-19:00'] > vm.CRIT['CR_min']
    add_summary('T8', '4.1', f'departure profile vs counts, {r["cordon"]} {r["direction"]}', 'cordon, hourly 06-19', 'CR > 0.6',
                f'CR {r["CR 06:00-19:00"]:.2f} (06-09 only: {r["CR 06:00-09:00 (3 hours)"]:.2f}); AM share of 06-19: survey {r["survey AM share of 06-19"]:.2f} vs counts {r["count AM share of 06-19"]:.2f}',
                vm.verdict(ok, explained=not ok),
                'Shape agreement of the whole-day profile; the survey puts more of the day in the morning peak than the road does (see the AM shares and the next table).', 'survey 2017/18; counts 2017-2023')

fig, axes = plt.subplots(2, 3, figsize=(14, 7), facecolor='white', sharex=True)
for ax, name in zip(axes.ravel(), CORDONS):
    for direction, col, ls in [('inbound', BLUE, '-'), ('outbound', ORANGE, '-')]:
        if (name, direction) in PROF:
            sv, c = PROF[(name, direction)]
            ax.plot(HOURS, c, color=col, lw=2, ls='-', label=f'counts, {direction}'); ax.plot(HOURS, sv, color=col, lw=1.4, ls='--', label=f'survey, {direction}')
    style_ax(ax, name, 'departure / count hour', 'share of 06:00-19:00'); ax.legend(frameon=False, fontsize=7)
save_fig(fig, 'mot_T8_car_profile_vs_counts.png')

## 2. Transit: survey departures against the RavKav boarding profile

In [5]:
rk = pd.read_csv('Output/ravkav_2025/boarding_profile_15min_2025.csv', index_col=0)
bins = np.arange(6, 9.0001, 0.25)
def qh(x):
    return np.array([x.loc[(x.t >= a) & (x.t < a + 0.25), 'w'].sum() for a in bins[:-1]])
rows = []; PT = {}
for nm, sel, col in [('bus (public bus + Metronit)', S.grp == 'bus', 'bus'), ('Metronit only (mode 5)', S.mainmode == 5, 'metronit')]:
    x = S[sel]; sv = qh(x); sv_share = sv / sv.sum(); c = rk[col].values.astype(float); c_share = c / c.sum()
    rows.append({'layer': nm, 'survey trips 06-09 (sampled)': int(((x.t >= 6) & (x.t < 9)).sum()), 'CR (12 quarter-hours)': vm.coincidence_ratio(sv_share, c_share),
                 'survey busiest hour share of 3 h': max(sv_share[i:i + 4].sum() for i in range(9)), 'RavKav busiest hour share of 3 h': max(c_share[i:i + 4].sum() for i in range(9)),
                 'survey busiest quarter-hour share': sv_share.max(), 'RavKav busiest quarter-hour share': c_share.max()})
    PT[nm] = (sv_share, c_share)
pt = pd.DataFrame(rows); pt.to_csv(f'{OUT}/T8_transit_profile_vs_ravkav.csv', index=False, float_format='%.4f'); pt.round(3)

,layer,survey trips 06-09 (sampled),CR (12 quarter-hours),survey busiest hour share of 3 h,RavKav busiest hour share of 3 h,survey busiest quarter-hour share,RavKav busiest quarter-hour share
0,bus (public bus + Metronit),1790,0.750,0.589,0.475,0.191,0.130
1,Metronit only (mode 5),180,0.742,0.526,0.433,0.185,0.115


In [6]:
for _, r in pt.iterrows():
    ok = r['CR (12 quarter-hours)'] > vm.CRIT['CR_min']
    add_summary('T8', '4.1', f'departure profile vs RavKav boardings, {r["layer"]}', 'quarter-hour 06-09', 'CR > 0.6',
                f'CR {r["CR (12 quarter-hours)"]:.2f}; busiest hour holds {r["survey busiest hour share of 3 h"]:.2f} of the three hours in the survey vs {r["RavKav busiest hour share of 3 h"]:.2f} in RavKav',
                vm.verdict(ok, explained=not ok), 'Departure time (survey) is not boarding time (RavKav) and the years differ (2017/18 vs 2025); the survey peak is sharper.', 'survey 2017/18; RavKav 2025 (42 Tuesdays)')
fig, axes = plt.subplots(1, 2, figsize=(11, 4), facecolor='white')
lab = [f'{int(a)}:{int(round((a % 1) * 60)):02d}' for a in bins[:-1]]
for ax, nm in zip(axes, PT):
    sv, c = PT[nm]; ax.plot(range(12), sv, color=BLUE, lw=2, label='survey, departures'); ax.plot(range(12), c, color=ORANGE, lw=2, label='RavKav 2025, boardings')
    ax.set_xticks(range(12)); ax.set_xticklabels(lab, rotation=60, fontsize=8); ax.legend(frameon=False, fontsize=8); style_ax(ax, nm, None, 'share of 06:00-09:00')
save_fig(fig, 'mot_T8_transit_profile_vs_ravkav.png')

**Reading.** The CR compares shapes. The table also gives the share of the morning that falls in the busiest hour in each source,
which is the number the peak-hour factors of step 20 rest on: the survey's departure profile is peakier than both independent profiles, so those
factors are an upper bound (METHODOLOGY section 8, caveat 18). A design-hour case on the observed profiles is the follow-up (handover C11).

## Summary of this notebook against the guideline

In [7]:
SUM = pd.DataFrame(SUMMARY); SUM.to_csv(f'{OUT}/summary_stage2{PSUF}.csv', index=False)
SUM[['test', 'guideline', 'item', 'level', 'criterion', 'result', 'status']]

,test,guideline,item,level,criterion,result,status
0,T8,4.1,"departure profile vs counts, Haifa city inbound","cordon, hourly 06-19",CR > 0.6,CR 0.78 (06-09 only: 0.72); AM share of 06-19:...,pass
1,T8,4.1,"departure profile vs counts, Haifa city outbound","cordon, hourly 06-19",CR > 0.6,CR 0.77 (06-09 only: 0.73); AM share of 06-19:...,pass
2,T8,4.1,"departure profile vs counts, Krayot inbound","cordon, hourly 06-19",CR > 0.6,CR 0.75 (06-09 only: 0.71); AM share of 06-19:...,pass
3,T8,4.1,"departure profile vs counts, Krayot outbound","cordon, hourly 06-19",CR > 0.6,CR 0.77 (06-09 only: 0.73); AM share of 06-19:...,pass
4,T8,4.1,"departure profile vs counts, Kiryat Ata + Zevu...","cordon, hourly 06-19",CR > 0.6,CR 0.79 (06-09 only: 0.82); AM share of 06-19:...,pass
5,T8,4.1,"departure profile vs counts, Kiryat Ata + Zevu...","cordon, hourly 06-19",CR > 0.6,CR 0.71 (06-09 only: 0.71); AM share of 06-19:...,pass
6,T8,4.1,"departure profile vs counts, Nazareth + Nof Ha...","cordon, hourly 06-19",CR > 0.6,CR 0.66 (06-09 only: 0.78); AM share of 06-19:...,pass
7,T8,4.1,"departure profile vs counts, Nazareth + Nof Ha...","cordon, hourly 06-19",CR > 0.6,CR 0.63 (06-09 only: 0.64); AM share of 06-19:...,pass
8,T8,4.1,"departure profile vs counts, Tirat Carmel inbound","cordon, hourly 06-19",CR > 0.6,CR 0.73 (06-09 only: 0.61); AM share of 06-19:...,pass
9,T8,4.1,"departure profile vs counts, Tirat Carmel outb...","cordon, hourly 06-19",CR > 0.6,CR 0.77 (06-09 only: 0.77); AM share of 06-19:...,pass
